# Italy–Ghana PINN pipeline — local notebook

Run from a local Jupyter/VS Code Python kernel. Setup uses the kernel interpreter and subprocess argument lists, not shell magics or PYTHONPATH assignments.

This is an orchestration refactor, not a replacement of the model implementation. Keep your current src scripts and data. Updated two-arm prepare_italy.py and sparsify.py interfaces are required.

Primary grids use common origins 45, 60, 75 and horizon 15; Ghana longer-window experiments should use separate files. Existing results must not be silently mixed with a different design. CPU/GPU speed must be measured locally.

WARNING: the previously supplied extract_rt.py computes a finite-difference proxy directly from I_proxy; it does not train a PINN. Its outputs must not be labelled PINN-inferred Rt. Do not treat cross-country Mann–Whitney results as causal evidence of missingness effects.


## 1. Runtime check

In [1]:
from pathlib import Path
import os, sys, subprocess, platform

# Set this to your repository folder if automatic discovery fails.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next((p for p in candidates if (p / "src" / "pinn.py").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Set PROJECT_ROOT to the folder containing src/pinn.py.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
NW = max(1, min(4, (os.cpu_count() or 1) - 1))

def run_script(script, *args):
    path = PROJECT_ROOT / "src" / script
    if not path.exists():
        raise FileNotFoundError(path)
    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = "1"
    env["MKL_NUM_THREADS"] = "1"
    command = [sys.executable, "-u", str(path), *map(str, args)]
    print("Running:", command, flush=True)
    subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)

print("Project:", PROJECT_ROOT)
print("Interpreter:", sys.executable)
print("System:", platform.platform())
print("CPU workers:", NW)


Project: D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-
Interpreter: c:\Users\BrothermanKNAN\anaconda3\python.exe
System: Windows-11-10.0.29683-SP0
CPU workers: 4


In [2]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


PyTorch: 2.12.1+cu126
CUDA available: True
GPU count: 1
0 NVIDIA GeForce MX250


## 2. Upload the project

Upload the extended pipeline archive (`thesis_pipeline.tar.gz` or an equivalent
`.zip` containing `src/` and `data/`). This must include BOTH the original
course-project files and the six new scripts (`prepare_italy.py`,
`synth_hospitalisation.py`, `sparsify.py`, `run_italy.py`, `equity_metrics.py`,
`stat_tests.py`), plus the raw Ghana HERA CSV and the raw Italy ISS
infections+hospitalisations CSV under `data/raw/`.

In [3]:
# Project files are read from PROJECT_ROOT; no Colab upload is needed.
print(PROJECT_ROOT)


D:\Anaconda Python Files\GitHub\PINN-Thesis-Framework-MPhil-2026-2027-


## 3. Benchmark: CPU vs GPU (both countries share one engine, so one check suffices)

Same measurement-not-assumption approach as the course project. Reuses the
Ghana daily-regime timing; Italy fits are cheaper per epoch since collocation
count is fixed regardless of series length, so if CPU wins here it wins there
too.

In [ ]:
import time, torch, numpy as np, pandas as pd, sys
sys.path.insert(0, "src")

from pinn import Config, ReducedSIRPINN

m = pd.read_csv("data/processed/ghana_national_master.csv", index_col=0, parse_dates=True)
I = m.I_proxy.to_numpy(float); scale = I.max(); origin = 180
td = torch.tensor(np.arange(origin) / origin, dtype=torch.float32).reshape(-1, 1)
yo = torch.tensor(I[:origin] / scale, dtype=torch.float32).reshape(-1, 1)

for dev in (["cpu", "cuda"] if torch.cuda.is_available() else ["cpu"]):
    c = Config(epochs_joint=60, n_collocation=6000, device=dev)
    t0 = time.time(); ReducedSIRPINN(c, tf=origin, scale_I=scale).fit_joint(td, yo)
    dt = time.time() - t0
    print(f"{dev:5s}: {dt:6.1f}s for 60 joint epochs -> "
          f"full 5000-epoch fit ~= {dt/60*5000/60:5.1f} min")


## 4. Validation gate — Millevoi Case 4 before all country arms

Before running Ghana, Italy-primary, or Italy-secondary experiments, validate the shared `ReducedSIRPINN` implementation against Millevoi et al.’s Case 4 benchmark. Every country arm uses the same core PINN class, so a failure here would invalidate all downstream country-specific results.

Published Case 4 targets from Millevoi et al.’s Table 4 are:

| Training approach | Published relative error: \(I(t)\) | Published relative error: \(R_t\) |
|---|---:|---:|
| Split | 0.1331 | 0.4744 |
| Joint | 0.1411 | 0.4961 |


**Decision rule:** if the split result is materially worse than the published \(err_I = 0.1331\) and \(err_{R_t} = 0.4744\), stop and diagnose the implementation before running Ghana or either Italy arm. At minimum, check the Case 4 input file, scaling convention, time normalization, random seed, mini-batching, collocation count, epoch settings, and whether the reference implementation’s model/data assumptions have been faithfully reproduced. The course-project validation is not a guarantee that the method will work on Ghana or Italy; it is a necessary implementation check showing that your PyTorch reimplementation behaves plausibly on the published synthetic benchmark.



In [ ]:
# Full validation: runs both split and joint against Millevoi Case 4.
run_script('pinn.py', 
           '--repo', 'reference', 
           '--out', 'results/tables/validation_case4.json', 
           '--mode', 'both', 
           '--epochs-joint', '5000', 
           '--epochs-split-data', '3000', 
           '--epochs-split-ode', '1000', 
           '--collocation', '6000', 
           '--seed', '34')


For free-tier planning, run the split validation first, since it is substantially cheaper and is the primary gate for the split-first experimental strategy:

In [ ]:
# First gate: validate the split formulation before any country-level grid.
run_script('pinn.py', 
           '--repo', 'reference', 
           '--out', 'results/tables/validation_case4.json', 
           '--mode', 'split', 
           '--epochs-split-data', '3000', 
           '--epochs-split-ode', '1000', 
           '--collocation', '6000', 
           '--seed', '34')


Only after this passes should you proceed with Ghana split and Italy-primary split grids. Then run the joint validation before investing in the much more expensive Ghana-joint and Italy-primary-joint experiments:

In [ ]:
# Second gate: validate joint before starting any joint country grid.
run_script('pinn.py', 
           '--repo', 'reference', 
           '--out', 'results/tables/validation_case4.json', 
           '--mode', 'joint', 
           '--epochs-joint', '5000', 
           '--collocation', '6000', 
           '--seed', '34')


This two-stage gate preserves the methodological requirement while fitting the split-first, interruption-aware workflow: validate the implementation required for each mode before spending compute on that mode.



## 5. Preprocessing — both countries + the synthetic hospitalisation channel

Runs `prepare_ghana.py` (unchanged) and the new `prepare_italy.py`, then generates:
- `ghana_hospitalisation_synth.csv` (SEIQHRS ablation channel, Dwomoh et al. parameters — explicitly a model-derived proxy, never treated as observed data downstream)
- `italy_primary_national_master.csv` (single-wave, r=6-valid window matched to the Millevoi et al. setup — the clean benchmark used for Objectives 1 & 2)
- `italy_secondary_national_master.csv` (full multi-wave range — Objective 3 regime-shift robustness arm only, kept separate throughout the pipeline)
- `italy_sparseNN.csv` files (20/30/40% controlled random dropout, via `sparsify.py`) generated separately for **both** the primary and secondary series, so downstream grids can pull sparsity regimes from either arm without collapsing the country label

This stage is cheap (<2 min total) and safe to re-run on a free-tier session without worrying about disconnects. Because `prepare_italy.py` now writes two labelled arms instead of one, double-check the `country` column on its outputs is set to `italy_primary`/`italy_secondary` before this feeds into the grid, equity, or stat-test stages — those downstream scripts refuse to run on an ambiguous or missing label.

In [ ]:
# NEW step — run this before prepare_italy.py
run_script('build_italy_national.py', 
           '--mode', 'bydate', 
           '--confirmed', 'data/raw/iss_bydate_italy_confirmed.csv', 
           '--hospitalisations', 'data/raw/iss_bydate_italy_hospitalizations.csv', 
           '--out', 'data/raw/italy_case_hosp.csv')


Thesis Primary benchmark (for Objectives 1 and 2): a single-wave, r=6-valid window — either Millevoi's original 90 days or a comparable early-2020 window — used for the core split-PINN validation and the Ghana comparability claims. This is your "clean" high-information benchmark.

In [ ]:
# PRIMARY arm: single-wave, r=6-valid window (Millevoi-comparable benchmark)
# Used for Objectives 1 & 2 -- core split-PINN validation and Ghana
# comparability claims.
run_script('prepare_italy.py', 
'--arm', 'primary', 
'--input', 'data/raw/italy_case_hosp.csv', 
'--outdir', 'data/processed')

# SECONDARY arm: full multi-wave range, ascertainment correction RESTRICTED
# to its valid window (see prepare_italy.py docstring). Labelled explicitly
# as a regime-shift robustness stress test for Objective 3 only -- NOT/
# presented as a clean data-richness comparison against Ghana.
run_script('prepare_italy.py', 
'--arm', 'secondary', 
'--ascertainment', 'restrict', 
'--input', 'data/raw/italy_case_hosp.csv', 
'--outdir', 'data/processed')

import json
print("\n=== PRIMARY arm audit ===")
print(json.dumps(json.load(open("data/processed/italy_primary_audit_summary.json")), indent=2))
print("\n=== SECONDARY arm audit ===")
print(json.dumps(json.load(open("data/processed/italy_secondary_audit_summary.json")), indent=2))


In [ ]:
# Ghana preprocessing. Italy preparation is in the preceding cell.
run_script("prepare_ghana.py", 
"--input", "data/raw/ghana_hera.csv", 
"--outdir", "data/processed")
run_script("synth_hospitalisation.py", 
           "--master", "data/processed/ghana_national_master.csv",
           "--params", "dwomoh_scenario5", 
           "--sensitivity", 
           "--out", "data/processed/ghana_hospitalisation_synth.csv")

# Preserve existing masks; do not regenerate a completed experiment's masks.
import pandas as pd
for arm in ("italy_primary", "italy_secondary"):
    path = Path(f"data/processed/{arm}_national_master.csv")
    frame = pd.read_csv(path, index_col=0)
    missing = [pct for pct in (20, 30, 40) if f"observed_sparse{pct}" not in frame.columns]
    if missing:
        run_script("sparsify.py", "--master", path, "--country", arm,
                   "--pct", *missing, "--inplace")


## 6. The grids — staged and sharded for free-tier survivability

### 6.1 Sizing (what to expect)

| Arm | Fits | Rough wall-clock, 2 vCPU (free) | 8 vCPU | 16 vCPU |
|---|---|---|---|---|
| Ghana split (unchanged) | 54 | ~100 min | ~25 min | ~13 min |
| Ghana joint (unchanged) | 54 | ~13 h | ~3.3 h | ~1.7 h |
| Italy PRIMARY split (4 regimes: daily, weekly, sparse20, sparse30) | 90 | ~90 min | ~22 min | ~11 min |
| Italy PRIMARY joint (4 regimes) | 90 | ~11 h | ~2.8 h | ~1.4 h |
| Italy SECONDARY split (4 regimes, multi-wave, optional Obj. 3 robustness) | 150 | ~150 min | ~37 min | ~19 min |
| Italy SECONDARY joint (4 regimes, optional) | 150 | ~18 h | ~4.6 h | ~2.3 h |
| Hospitalisation ablation (Ghana, split only, ×2 for with/without H) | 54 | ~100 min | ~25 min | ~13 min |

**On free tier (2 vCPU, no background execution): run split arms only in any one sitting.** Joint arms must go on a paid/High-RAM session with background execution, or be broken into per-origin chunks across several free sessions (Section 6.4). The SECONDARY Italy arm is larger and longer-running because it spans multiple epidemic waves rather than one; since it feeds only the Objective 3 robustness check (not the core equity comparison), it is lowest priority — run it last, and only once PRIMARY is done and validated.

### 6.2 Split-first policy (unchanged principle, now for three arms)

Always run every `split` arm before any `joint` arm — split is ~6× cheaper and gives you a working results table even if the session dies before joint finishes. With three arms in play, the recommended order is:

1. Ghana split → Italy PRIMARY split (core comparison, cheapest first).
2. Ghana joint → Italy PRIMARY joint (once split results confirm the pipeline works).
3. Italy SECONDARY split, then joint — only after PRIMARY is complete, since SECONDARY is an optional Objective 3 robustness arm, not required for the main equity result.

This ordering ensures a session dying early still leaves you with the two arms your central equity claim depends on (Ghana + Italy PRIMARY), rather than burning free-tier hours on the robustness-only SECONDARY arm before the core comparison is even finished.

In [ ]:
import os
# NW is configured in setup; change there if needed.
print(f"Using {NW} workers.")

# --- Ghana split (unchanged grid) ---
run_script('run_parallel.py', 
           '--workers', NW, 
           '--modes', 'split', 
           '--origins', '45', '60', '75', 
           '--regimes', 'daily', 'weekly', 'mask', 
           '--seeds', '1', '2', '3', 
           '--horizon', '15', 
           '--out', 'results/tables/grid_ghana_split.csv')



In [ ]:
## In case you would want to run the Ghana split grid on GPU, 
# you can use the following command. It is not necessary for the main experiments, 
# but it can be useful for testing or if you want to speed up the computation.

"""
This runner leaves device selection to Config(device="auto"), 
so it can use CUDA when available. It runs fits sequentially and, 
in the supplied version, writes the results CSV only after the whole command completes—not after every fit
"""


run_script(
    "run_ghana.py",
    "--master", "data/processed/ghana_national_master.csv",
    "--out", "results/tables/grid_ghana_split_gpu.csv",
    "--origins", "45", "60", "75",
    "--regimes", "daily", "weekly", "mask",
    "--modes", "split",
    "--seeds", "1", "2", "3",
    "--horizon", "15",
)

In [ ]:

# --- Italy PRIMARY arm split grid (single-wave, Millevoi-comparable) ---
run_script('run_italy.py', 
           '--master', 'data/processed/italy_primary_national_master.csv', 
           '--out', 'results/tables/italy_primary_grid_split.csv', 
           '--origins', '45', '60', '75', ''
           '--horizon', '15', 
           '--regimes', 'daily', 'weekly', 'sparse20', 'sparse30', 'sparse40', 
           '--modes', 'split', 
           '--seeds', '1', '2', '3')



# Objective 3 supplementary robustness read (regime-shift stress test).
# NOT part of the primary equity/comparability claim above.

In [ ]:
# --- Italy SECONDARY arm split grid (full range, regime-shift stress test) ---
run_script('run_italy.py',
            '--master', 'data/processed/italy_secondary_national_master.csv',
              '--out', 'results/tables/italy_secondary_grid_split.csv', 
              '--origins', '75', '90', '100', '150', '300', 
              '--horizon', '15', 
              '--regimes', 'daily', 'weekly', 'sparse20', 'sparse30', 'sparse40', 
              '--modes', 'split', 
              '--seeds', '1', '2', '3')


### 6.3 Joint arms (paid tier / background execution recommended)

Enable **Runtime → Background execution** first if available, or the joint
runs below will die when the tab closes. If you are on free tier without
background execution, use the chunked plan in 6.4 instead of this cell.

In [ ]:
import os
# NW is configured in setup; change there if needed.
## --- Ghana joint (unchanged grid) --- '

print(f"Using {NW} workers.")

run_script('run_parallel.py', '--workers', NW, ''
'--modes', 'joint', 
'--origins', '45', '60', '75',
 '--regimes', 'daily', 'weekly', 'mask', 
 '--seeds', '1', '2', '3', 
 '--horizon', '15',
   '--out', 'results/tables/grid_ghana_joint.csv')





In [ ]:
import os
# NW is configured in setup; change there if needed.
## --- Italy PRIMARY arm joint grid (single-wave, Millevoi-comparable) ---
run_script('run_italy.py', '--master', 'data/processed/italy_primary_national_master.csv', 
           '--out', 'results/tables/italy_primary_grid_joint.csv', 
           '--origins', '45', '60', '75', '--horizon', '15', 
           '--regimes', 'daily', 'weekly', 'sparse20', 'sparse30', 'sparse40', 
           '--modes', 'joint', 
           '--seeds', '1', '2', '3')


In [ ]:
## --- Italy SECONDARY arm joint grid (full range, regime-shift stress test) ---
run_script('run_italy.py', 
           '--master', 'data/processed/italy_secondary_national_master.csv', 
           '--out', 'results/tables/italy_secondary_grid_joint.csv', 
           '--origins', '90', '100', '150', '300', 
           '--horizon', '15', 
           '--regimes', 'daily', 'weekly', 'sparse20', 'sparse30', 'sparse40', 
           '--modes', 'joint', 
           '--seeds', '1', '2', '3')


### 6.4 Free-tier fallback: chunk the joint grid by origin across sessions

If background execution is unavailable, split the joint grid by origin and
run one or two origins per session, appending to the same CSV. Re-run this
cell each session with the NEXT_ORIGINS list updated; already-completed
origins are skipped automatically via the `--resume` check against existing
rows in the output CSV (see `run_stages.py`'s resume-safe pattern, reused
here for a single origin at a time).

In [ ]:
# Archived cloud/chunk examples omitted. Use the main local grid cells.


### 6.5 Hospitalisation ablation (Ghana split only — cheap, safe for free tier)

Compares infection-only training against infection+synthetic-H training,
using the SEIQHRS-derived channel from Stage 5. This directly tests whether
a second observed channel helps the way it does for Italy's real
hospitalisation series in Millevoi et al.'s Cases 6–7.

**Note:** `run_ghana.py`/`run_parallel.py` as inherited only support a
single-channel `I_proxy` target. Running the two-channel ablation requires
either (a) extending `run_ghana.py` with an `--aux-channel` flag pointing at
`H_synthetic_seiqhrs`, mirroring how Millevoi et al. add hospitalisations as
a second loss term, or (b) a standalone ablation script. This is flagged
here as **implementation work still to be done on `pinn.py`'s loss function**
before this stage can run — it is not yet wired into the shared CLI.

In [ ]:
# Disabled: the shared model does not implement a hospitalisation loss.
# Generating a synthetic H channel does not enable two-channel fitting.


## 7. Merge grids, then equity metrics and statistical tests

In [ ]:
import pandas as pd, os

def load_parts(paths):
    parts = [pd.read_csv(p) for p in paths if os.path.exists(p)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

ghana_grid = load_parts(["results/tables/grid_ghana_split.csv",
                          "results/tables/grid_ghana_joint.csv"])
ghana_grid["country"] = "ghana"
ghana_grid.to_csv("results/tables/grid_ghana.csv", index=False)

italy_primary = load_parts(["results/tables/italy_primary_grid_split.csv",
                             "results/tables/italy_primary_grid_joint.csv"])
italy_primary["country"] = "italy_primary"

italy_secondary = load_parts(["results/tables/italy_secondary_grid_split.csv",
                               "results/tables/italy_secondary_grid_joint.csv"])
italy_secondary["country"] = "italy_secondary"

# Objectives 1 & 2 (clean comparability) use italy_primary ONLY against Ghana.
italy_grid_for_equity = italy_primary.copy()
italy_grid_for_equity.to_csv("results/tables/italy_grid.csv", index=False)



In [ ]:
# Equity metrics: main Ghana vs Italy-primary comparison
run_script('equity_metrics.py', 
           '--grid', 'results/tables/grid_ghana.csv', 
           '--italy-grid', 'results/tables/italy_grid.csv', 
           '--out', 'results/tables/equity_summary_primary.csv')


In [ ]:
# Statistical tests: main Ghana vs Italy-primary comparison
run_script('stat_tests.py', 
           '--grid', 'results/tables/grid_ghana.csv', 
           '--italy-grid', 'results/tables/italy_grid.csv', 
           '--out', 'results/tables/significance_primary.csv', 
           '--italy-arm', 'primary')


## 8. Rt validity — both countries (extends `extract_rt.py`)

Ghana: checked against Dwomoh et al.'s Re estimates (unchanged from course
project). Italy: checked against the ISS regional/national Rt series
(`iss_rt_*.csv`) uploaded alongside the raw data.

In [ ]:
# Ghana: full Ghana analysis series
run_script('extract_rt.py', 
           '--master', 'data/processed/ghana_national_master.csv', 
           '--country', 'ghana', 
           '--outdir', 'results/rt')

# Italy PRIMARY: single-wave, main comparison arm
run_script('extract_rt.py', 
           '--master', 'data/processed/italy_primary_national_master.csv', 
           '--country', 'italy_primary', 
           '--outdir', 'results/rt')



## 9. Two-country data-quality figure and degradation plots

In [ ]:
run_script('fig_dataquality.py', )


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

r = pd.read_csv("results/tables/grid_ghana.csv"); r["country"] = "ghana"
ri = pd.read_csv("results/tables/italy_grid.csv")
if "country" not in ri.columns: ri["country"] = "italy"
combined = pd.concat([r, ri], ignore_index=True)
pinn = combined[combined.method.str.startswith("pinn")]

fig, ax = plt.subplots(1, 2, figsize=(14, 6.5))
for country, marker in [("ghana", "o"), ("italy", "s")]:
    sub = pinn[(pinn.country == country) & (pinn.method == "pinn_split")]
    order = [g for g in ["daily", "weekly", "mask", "sparse20", "sparse30", "sparse40"] if g in sub.regime.unique()]
    med = [sub[sub.regime == g].mase.median() for g in order]
    ax[0].plot(order, med, marker=marker, label=f"{country} (split)")
ax[0].set_ylabel("Median MASE"); ax[0].set_xlabel("Observation regime")
ax[0].set_title("Split-schedule degradation, Italy vs Ghana"); ax[0].legend(frameon=False)
ax[0].tick_params(axis="x", rotation=30)

eq = pd.read_csv("results/tables/equity_summary_primary.csv")
piv = eq.groupby(["country", "regime"])["mase_gap_pct"].median().unstack("country")
piv.plot(kind="bar", ax=ax[1])
ax[1].set_ylabel("Median MASE gap vs daily (%)"); ax[1].set_title("Equity gap by regime")
for a_ in ax: a_.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("results/figures/fig_italy_ghana_equity.png", dpi=180); plt.show()


# Objective 3 supplementary robustness read (regime-shift stress test).
# NOT part of the primary equity/comparability claim above.

In [ ]:
# Objective 3 robustness supplement keeps italy_secondary separate, never
# silently pooled with the primary/Ghana comparability claim.
italy_secondary.to_csv("results/tables/italy_grid_secondary_robustness.csv", index=False)

print("Ghana:", ghana_grid.shape, " Italy primary:", italy_primary.shape,
      " Italy secondary (robustness-only):", italy_secondary.shape)


In [ ]:
# Equity: also generate a separate Italy-secondary robustness output
run_script('equity_metrics.py', 
           '--grid', 'results/tables/grid_ghana.csv', 
           '--italy-grid', 'results/tables/italy_grid.csv', 
           '--out', 'results/tables/equity_summary_secondary.csv', 
           '--include-secondary')


In [ ]:
# Objective 3 supplementary robustness read (regime-shift stress test).
# NOT part of the primary equity/comparability claim above.
sec = pd.read_csv("results/tables/italy_grid_secondary_robustness.csv")
sec_pinn = sec[sec.method.str.startswith("pinn")]
print(sec_pinn.groupby(["method", "regime"])["mase"].median().round(3).to_string())
print("\nNote: italy_secondary spans multiple COVID-19 waves/variants and a "
      "restricted (partially NaN) ascertainment correction -- interpret as "
      "a regime-shift robustness signal, not a like-for-like data-richness "
      "comparison against Ghana or italy_primary.")


In [ ]:
# Italy SECONDARY: optional multi-wave Objective-3 robustness arm
run_script('extract_rt.py', 
           '--master', 'data/processed/italy_secondary_national_master.csv', 
           '--country', 'italy_secondary', 
           '--outdir', 'results/rt')


## 10. Download everything

In [ ]:
import shutil
archive = shutil.make_archive(str(PROJECT_ROOT / "thesis_results_local"), "zip", PROJECT_ROOT / "results")
print("Results archive:", archive)
